# REDGARDEN Arena Bot AI — PFSP League Training (fast-forward, Colab)

Runs on Google Colab (GPU or CPU). Launches the same real AlphaStar-style league
REDGARDEN's `scripts/run_league.sh` already runs locally — three concurrent roles
(Main / Main Exploiter / League Exploiter, NORTHSTAR §25.4.1), each training PPO against a
shared, permanent, PFSP-weighted opponent registry — just pointed at your own Google Drive so
checkpoints, the league registry, and each checkpoint directory's own real git-lfs history
(`scripts/checkpoint_git_sync.py`, S533) all survive a Colab session reset instead of living in
the ephemeral container filesystem.

Founder real-time: "...begin work on pure pure pure RL auto curiculum PFSP fast play fast
forward league play via a colab skrip."

**Honest, current status (2026-09-22):** this notebook's own driver (`scripts/colab_train_league.py`)
is real and structurally verified — it correctly builds `libarena_training.so`, launches all
three role processes with the right arguments, and wires up git-lfs checkpoint sync. A full,
live end-to-end PPO training run has **not** been verified in this development sandbox: a
real, pre-existing, unrelated segfault was found in `stable-baselines3`'s own import chain in
that specific local Python/library combination (reproduces even with zero REDGARDEN-side flags
set, crashing inside `PyObject_Malloc` during nested imports before any REDGARDEN code runs at
all) — not something introduced by this work, and very likely specific to that one sandboxed
environment rather than Colab's own, different Python build. First real Colab run of this
notebook is the actual live verification; if a similar crash shows up here too, that's real,
new information worth reporting, not something this notebook papers over.

**Every change to how league training works ships as commits to `scripts/colab_train_league.py`**
— re-running the bootstrap cell below always executes the current version, no re-pasting.

In [ ]:
# === REDGARDEN PFSP league — reusable bootstrap cell ===
# This cell is the only thing you ever need to paste into Colab. It mounts
# Drive (approve the OAuth prompt when it appears), then pulls the latest
# training logic from git and runs it.

from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess

REPO_URL = 'https://github.com/emilyspringerton/REDGARDEN.git'
REPO_DIR = '/content/REDGARDEN'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.environ['REPO_DIR'] = REPO_DIR
# Adjust DRIVE_FOLDER here only if your Drive layout differs from the default.
os.environ.setdefault('DRIVE_FOLDER', '/content/drive/MyDrive/redgarden-league')

# --- Optional tuning (uncomment/edit any of these before running) ---
# os.environ['RL_TEAM_SIZE'] = '3'            # heroes per side, default 3 (3v3)
# os.environ['RL_TOTAL_TIMESTEPS'] = '500000' # per-role budget, matching run_league.sh's own default
#
# --- Optional: push checkpoint git-lfs history to a real remote (off by default -- checkpoints
#     still get a real local git-lfs history on Drive either way, S533's own design) ---
# os.environ['CHECKPOINT_GIT_REMOTE'] = 'git@github.com:emilyspringerton/<some-model-repo>.git'
# os.environ['CHECKPOINT_GIT_SSH_KEY'] = '/content/drive/MyDrive/.ssh/id_ed25519'

subprocess.run(
    ['python3', 'scripts/colab_train_league.py'],
    cwd=REPO_DIR, check=True,
)

## Next steps

This cell runs for a real, substantial amount of wall-clock time (three concurrent PPO
trainers) — tail `<DRIVE_FOLDER>/rl_league_<role>.log` from another Colab cell or the Drive web
UI to watch progress while it's running.

After a run (or while one is still going, since checkpoints sync as they save):
1. Checkpoints live in `<DRIVE_FOLDER>/rl_league_checkpoints_<role>/`, each its own real
   git-lfs repo (`scripts/checkpoint_git_sync.py`) — `git log` inside any one of them for a
   real, restorable history, independent of whether a remote was configured.
2. The shared league registry lives in `<DRIVE_FOLDER>/rl_league/` — point a fresh Colab run's
   own `--league-dir` at the same path (default, if `DRIVE_FOLDER` is unchanged) to resume
   training against the same growing league rather than starting over.
3. `--skip-export` is passed by `scripts/colab_train_league.py` — a team-mode policy has no
   live consumer yet (see `scripts/rl_train_team.py`'s own module doc comment), so no embedded-C
   export step runs here.